# M3 decision reports on Kaggle

Runs `serpentguard.decision.decision_report` on every training run whose kernel output is attached to this notebook, on
the CPU. For each run it fits the verdict thresholds on the calibration outputs, reports the verdicts on the
validation and unknown-pool outputs, and writes `decision_metrics.json`. A summary table of the main verdict numbers
goes to `decision_summary.csv`.

It runs on Kaggle rather than locally so the saved outputs (`*.npz`) never have to be downloaded. A kernel can read
only its own account's private outputs, so each account runs its own copy. No GPU is used.

In [1]:
RUN_KERNELS = ['vikas25s/sg-m3-e2c', 'vikas25s/sg-m3-e4n0']
QUALITY_KERNEL = None   # optional "owner/slug" whose output holds quality.csv (Task 10), for the poor_image reason

In [2]:
import subprocess
import sys

SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt, a Global Constraint: every metric uses it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, f"scikit-learn {sklearn.__version__} is loaded"

In [3]:
import json
import shutil
from pathlib import Path

import kagglehub
import pandas as pd

OWNER = "puneetbajajiitm"
CODE = Path("/kaggle/input/serpentguard-code/src")
sys.path.insert(0, str(CODE if CODE.exists() else Path("../src").resolve()))

from serpentguard.decision import decision_report

RUN_FILES = ("val_outputs.npz", "calib_outputs.npz", "unknown_outputs.npz", "prototypes.npz", "final_metrics.json",
             "config.yaml")
OUT = Path("/kaggle/working/decision")

data_root = Path(kagglehub.dataset_download(f"{OWNER}/serpentguard-indian-snakes-clean"))
classes = pd.read_csv(data_root / "classes.csv")
metadata = pd.read_csv(data_root / "metadata.csv", usecols=["path", "split"])


def kernel_output(handle):
    return Path(kagglehub.notebook_output_download(handle))


def run_folders(root):
    return sorted(p.parent for p in root.rglob("val_outputs.npz") if (p.parent / "calib_outputs.npz").exists())


quality = None
if QUALITY_KERNEL:
    found = sorted(kernel_output(QUALITY_KERNEL).rglob("quality.csv"))
    quality = pd.read_csv(found[0]) if found else None
    print("quality table:", found[0] if found else "not found")

In [4]:
rows = []
for handle in RUN_KERNELS:
    for source in run_folders(kernel_output(handle)):
        run = OUT / source.name
        run.mkdir(parents=True, exist_ok=True)
        for name in RUN_FILES:
            if (source / name).exists():
                shutil.copy2(source / name, run / name)
        report = decision_report(run, classes, metadata=metadata, quality=quality)
        for name in RUN_FILES:
            if name.endswith(".npz"):
                (run / name).unlink()
        metrics, fit = report["verdict_metrics"], report["fit"]
        rows.append({"run": source.name, "kernel": handle,
                     "who_dangerous_recall": metrics["who_dangerous_recall"], "false_alarm": metrics["false_alarm"],
                     "venomous_recall": metrics["venomous_recall"],
                     "o3_all": metrics["o3"]["dangerous_as_not_listed_all"], "abstain_rate": metrics["abstain_rate"],
                     "unknown_flagged": report["u1"]["groups"]["all"]["flagged_at_tau"],
                     "unknown_auroc_msp": report["u1"]["groups"]["all"]["open_set"]["msp"]["auroc"],
                     "theta_d": report["thresholds"]["theta_d"], "theta_who": report["thresholds"]["theta_who"],
                     "calibration_recall": fit["who_dangerous_recall"]})
        print(f"{source.name}: WHO recall {metrics['who_dangerous_recall']:.3f}, false alarms "
              f"{metrics['false_alarm']:.3f}, unknown flagged {rows[-1]['unknown_flagged']:.3f}")
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "decision_summary.csv", index=False)
assert len(summary), "no run folders found in the attached kernel outputs"
summary

E2c: WHO recall 0.949, false alarms 0.421, unknown flagged 0.040
E3: WHO recall 0.954, false alarms 0.379, unknown flagged 0.060
E4: WHO recall 0.949, false alarms 0.359, unknown flagged 0.055
N0_s1: WHO recall 0.949, false alarms 0.376, unknown flagged 0.040


,run,kernel,who_dangerous_recall,false_alarm,venomous_recall,o3_all,abstain_rate,unknown_flagged,unknown_auroc_msp,theta_d,theta_who,calibration_recall
0,E2c,vikas25s/sg-m3-e2c,0.949320,0.421282,0.937354,0.028625,0.072591,0.039911,0.543538,0.050467,0.608173,0.952297
1,E3,vikas25s/sg-m3-e2c,0.954481,0.378913,0.940979,0.026748,0.075273,0.059867,0.653647,0.053175,0.805916,0.950530
2,E4,vikas25s/sg-m3-e4n0,0.948850,0.359272,0.943567,0.029094,0.085359,0.055432,0.648865,0.068726,inf,0.952297
3,N0_s1,vikas25s/sg-m3-e4n0,0.948850,0.376279,0.938390,0.029564,0.082269,0.039911,0.638300,0.053645,0.616876,0.950530
